In [ ]:
import corner
import time
import os
import sys
import json
from datetime import datetime
from os.path import expanduser

home = expanduser("~/")
srcdir = os.path.join(home, 'gigalens/src/')

sys.path.insert(0, srcdir)
sys.path.insert(0, home+'/GIGALens-Code')
print('Harry GIGALENS IMPLEMENTATION')

import jax
# jax.config.update("jax_enable_x64", True)

from gigalens.jax.inference import HarryModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.model import PhysicalModel
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import tensorflow_probability.substrates.jax as tfp
from jax import random
from jax import numpy as jnp
import numpy as np
import optax
from matplotlib import pyplot as plt
from astropy.io import fits
from astropy.visualization import simple_norm
from corner import corner
tfd = tfp.distributions
import pickle
import helpers
from helpers import *
import pandas as pd

#* For slurm jobs
# jax.distributed.initialize()

#* For local, single-gpu testing
# jax.distributed.initialize(
#     coordinator_address="localhost:12346",
#     num_processes=1,
#     process_id=0
# )

#%%


In [ ]:
save_dir = os.path.join(home, f"GIGALens-Code/benchmarking_results")

jax.experimental.multihost_utils.sync_global_devices("run_start")
kernel = np.load(os.path.join(srcdir, 'gigalens/assets/psf.npy')).astype(np.float32)
observed_img = np.load(os.path.join(srcdir, 'gigalens/assets/demo.npy'))

#* Load 100 testing systems
# systems_dir = os.path.join(home, "GIGALens-Code/SystemSaves")
# f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
# keys = f.files
# observed_imgs = jnp.array([f[key] for key in keys])
# observed_img = observed_imgs[4]

prior = helpers.make_default_prior()

phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=1, kernel=kernel) 


model_seq = HarryModellingSequence(phys_model, prob_model, sim_config)

map_results = MAPResults.load(os.path.join(home, "GIGALens-Code", "testing", "svi_ada_sys4_harry"), model_seq)

opt_sgd = optax.sgd(learning_rate=1e-5)

schedule_fn = optax.polynomial_schedule(init_value=-1e-6, end_value=-3e-3,
                                    power=2, transition_steps=300)
opt_adam = optax.chain(
    optax.scale_by_adam(),
    optax.scale_by_schedule(schedule_fn),
)

opt_adabelief = optax.adabelief(1e-4, b1=0.97, b2=0.99)

pipeline_config = PipelineConfig(
    steps=["MAP", "SVI"], svi_optimizer=opt_adabelief)#, svi_start=map_results.best_z,)


In [ ]:
results = run_pipeline(model_seq, pipeline_config)

In [ ]:
svi_results = results["SVI"]
map_results = results["MAP"]
# map_results.save(os.path.join(home, "GIGALens-Code", "testing", "svi_ada_sys4_harry"))
# svi_results.save(os.path.join(home, "GIGALens-Code", "testing", "svi_ada_sys4_harry"))

fig, axs = plt.subplots(1, 2)
plot_loss_histories(fig, axs, map_results.MAP_chisq_hist, svi_results.SVI_loss_hist)
plt.show()

In [ ]:
# map_results.save(os.path.join(home, "GIGALens-Code", "benchmarking_results", "benchmark_starts"))
# svi_results.save(os.path.join(home, "GIGALens-Code", "benchmarking_results", "benchmark_starts"))

In [ ]:
# svi_results = SVIResults.load(os.path.join(home, "GIGALens-Code", "testing", "svi_ada_sys4_harry"), model_seq)


pipeline_config_hmc = PipelineConfig(
    steps=["HMC"], qz=svi_results.qz)

HMC_results = run_pipeline(model_seq, pipeline_config_hmc)["HMC"]

In [ ]:
lens_sim = LensSimulator(phys_model, sim_config, bs=1)

results = {"MAP": map_results, "SVI": svi_results, "HMC": HMC_results}
display_results(results, observed_img, lens_sim)

In [ ]:
HMC_results.HMC_rhat